# 00 — Exploración del proxy LiteLLM

Notebook para la **Fase 0** del plan: ver qué modelos expone el proxy de la universidad y elegir los valores de
`LLM_MODEL`, `EMBEDDING_MODEL` y `EMBEDDING_DIM` para el `.env`.

Qué verifica:
1. Que las variables `LITELLM_BASE_URL` y `LITELLM_API_KEY` estén cargadas.
2. Lista de modelos (`/models`) y, si el proxy lo permite, su info detallada (`/model/info`: tipo, contexto, dimensión).
3. Ping a los modelos de chat (latencia + que respondan).
4. **Dimensión** de cada modelo de embeddings (para crear el índice de Pinecone).
5. Que el modelo de chat elegido soporte **salida estructurada** (Pydantic) y **tool calling** con LangChain — lo usan los dos agentes.
6. Resumen con los valores para pegar en el `.env`.

> Las llamadas son mínimas (prompts de una línea) para no gastar presupuesto de la clave.
> Antes de commitear, limpiá los outputs si no querés versionar nombres de modelos / errores del proxy.

In [ ]:
import os
import time
import json

import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)

BASE_URL = os.environ.get("LITELLM_BASE_URL", "").rstrip("/")
API_KEY = os.environ.get("LITELLM_API_KEY", "")

assert BASE_URL, "Falta LITELLM_BASE_URL en el .env"
assert API_KEY, "Falta LITELLM_API_KEY en el .env"

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
# /model/info es un endpoint propio de LiteLLM que vive en la raíz (sin /v1)
ROOT_URL = BASE_URL[:-3] if BASE_URL.endswith("/v1") else BASE_URL

print("LITELLM_BASE_URL cargada:", bool(BASE_URL), "| termina en /v1:", BASE_URL.endswith("/v1"))
print("LITELLM_API_KEY cargada:", bool(API_KEY), f"({len(API_KEY)} chars)")

## 1. Modelos disponibles (`/models`)

In [ ]:
resp = requests.get(f"{BASE_URL}/models", headers=HEADERS, timeout=30)
print("HTTP", resp.status_code)
resp.raise_for_status()

model_ids = sorted(m["id"] for m in resp.json().get("data", []))
print(f"{len(model_ids)} modelos:")
for m in model_ids:
    print(" -", m)

## 2. Info detallada (`/model/info`)

Endpoint específico de LiteLLM. Puede estar deshabilitado para claves de usuario; si falla, clasificamos por nombre y probando.

In [ ]:
model_info: dict[str, dict] = {}
for url in (f"{ROOT_URL}/model/info", f"{BASE_URL}/model/info", f"{ROOT_URL}/v1/model/info"):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.ok and "data" in r.json():
            for entry in r.json()["data"]:
                info = entry.get("model_info") or {}
                model_info[entry["model_name"]] = {
                    "mode": info.get("mode"),
                    "max_input_tokens": info.get("max_input_tokens"),
                    "max_output_tokens": info.get("max_output_tokens"),
                    "output_vector_size": info.get("output_vector_size"),
                    "supports_function_calling": info.get("supports_function_calling"),
                    "supports_response_schema": info.get("supports_response_schema"),
                    "input_cost_per_token": info.get("input_cost_per_token"),
                    "output_cost_per_token": info.get("output_cost_per_token"),
                }
            print("OK desde", url.replace(ROOT_URL, "<ROOT>"))
            break
        print(r.status_code, "en", url.replace(ROOT_URL, "<ROOT>"))
    except Exception as e:
        print("Error:", type(e).__name__, e)

for name, info in sorted(model_info.items()):
    print(f"{name:40s}", json.dumps(info))

## 3. Clasificar modelos: chat vs embeddings

Usa `mode` de `/model/info` si está; si no, una heurística por nombre (`embed` en el nombre).

In [ ]:
def is_embedding(model: str) -> bool:
    mode = (model_info.get(model) or {}).get("mode")
    if mode:
        return mode == "embedding"
    return "embed" in model.lower()

embedding_models = [m for m in model_ids if is_embedding(m)]
chat_models = [m for m in model_ids if not is_embedding(m)]
print("Chat:", chat_models)
print("Embeddings:", embedding_models)

## 4. Ping a los modelos de chat

Una llamada corta por modelo. Primero **sin** `temperature` (algunos modelos nuevos la rechazan) y después
probando `temperature=0`, que es lo que conviene para extracción y generación de código.
`max_tokens` no es tan bajo porque los modelos de razonamiento (gpt-5*) gastan tokens pensando antes de responder.
Si hay muchos modelos, limitá la lista con `CHAT_MODELS_TO_TEST`.

In [ ]:
from langchain_openai import ChatOpenAI

CHAT_MODELS_TO_TEST = [m for m in chat_models if m != "all-proxy-models"]  # p. ej. ["gpt-4.1-mini"]


def make_llm(model: str, **kw) -> ChatOpenAI:
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, model=model, timeout=120, max_retries=0, **kw)


def short_err(e: Exception) -> str:
    return f"{type(e).__name__}: {str(e)[:160]}"


chat_results = {}
for model in CHAT_MODELS_TO_TEST:
    res = {}
    t0 = time.perf_counter()
    try:
        out = make_llm(model, max_tokens=300).invoke("Respondé solo con la palabra: OK")
        res.update(ok=True, latency_s=round(time.perf_counter() - t0, 2), reply=out.text.strip()[:20],
                   tokens=out.usage_metadata and out.usage_metadata.get("total_tokens"))
        try:
            make_llm(model, max_tokens=300, temperature=0).invoke("Respondé solo con la palabra: OK")
            res["temperature_0"] = True
        except Exception:
            res["temperature_0"] = False
    except Exception as e:
        res.update(ok=False, error=short_err(e))
    chat_results[model] = res
    print(f"{model:28s}", res)

## 5. Dimensión de los modelos de embeddings

La dimensión define el índice de Pinecone. También probamos `check_embedding_ctx_length=False`
(evita que LangChain tokenice con `tiktoken` asumiendo un modelo de OpenAI, que a veces falla con modelos no-OpenAI).

In [ ]:
from langchain_openai import OpenAIEmbeddings

emb_results = {}
for model in embedding_models:
    res = {}
    for check_ctx in (True, False):
        emb = OpenAIEmbeddings(base_url=BASE_URL, api_key=API_KEY, model=model,
                               check_embedding_ctx_length=check_ctx, max_retries=0)
        t0 = time.perf_counter()
        try:
            v = emb.embed_query("Ingresos anuales por región, en millones de pesos")
            docs = emb.embed_documents(["tabla de desempleo", "tabla de población"])
            res[f"check_ctx={check_ctx}"] = {"ok": True, "dim": len(v), "batch_dims": [len(d) for d in docs],
                                             "latency_s": round(time.perf_counter() - t0, 2)}
        except Exception as e:
            res[f"check_ctx={check_ctx}"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    emb_results[model] = res
    print(model)
    for k, val in res.items():
        print("   ", k, val)

### 5.1 Sanity check semántico

Similitud coseno: frases relacionadas deberían dar más alto que frases no relacionadas.

In [ ]:
import math

def cos(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

query = "¿Qué región tuvo la tasa de desempleo más alta en 2023?"
docs = {
    "relacionada": "Tasa de desempleo por región y año, 2019-2023 (%)",
    "no relacionada": "Ingresos de la empresa por línea de producto, en millones de USD",
}
for model in embedding_models:
    if not any(r.get("ok") for r in emb_results[model].values()):
        continue
    check_ctx = bool(emb_results[model]["check_ctx=True"].get("ok"))
    emb = OpenAIEmbeddings(base_url=BASE_URL, api_key=API_KEY, model=model, check_embedding_ctx_length=check_ctx)
    qv = emb.embed_query(query)
    dv = emb.embed_documents(list(docs.values()))
    print(model, {k: round(cos(qv, v), 3) for k, v in zip(docs, dv)})

## 6. Comparación de candidatos para `LLM_MODEL`

Para cada candidato probamos lo que van a necesitar los agentes:
- **Salida estructurada** con Pydantic (`json_schema` y `function_calling`): enriquecimiento de metadatos, plan de consulta, selección de tablas.
- **Tool calling** (`bind_tools`).
- **Generación de código**: que devuelva un programa pandas que imprima una línea JSON, ejecutarlo en un subproceso y verificar el valor (esperado: 22.5).

Editá `CANDIDATES` para probar otros modelos. Si ya tenés `LLM_MODEL` en el `.env`, se agrega a la lista.

In [ ]:
import re
import subprocess
import sys
import tempfile
import pathlib

from pydantic import BaseModel, Field
from langchain_core.tools import tool

working_chat = [m for m, r in chat_results.items() if r.get("ok")]
CANDIDATES = ["gpt-4.1-mini", "gpt-4.1", "gpt-5.4-mini", "gpt-5.4", "claude-haiku-4-5", "claude-sonnet-4-5"]
if os.environ.get("LLM_MODEL"):
    CANDIDATES.insert(0, os.environ["LLM_MODEL"])
CANDIDATES = [m for m in dict.fromkeys(CANDIDATES) if m in working_chat]
print("Candidatos:", CANDIDATES)


class ColumnInfo(BaseModel):
    name: str = Field(description="Nombre exacto de la columna")
    description: str
    dtype_unit: str = Field(description="Tipo y unidad, ej. 'float, %'")


class TableMetadata(BaseModel):
    name: str
    description: str
    columns: list[ColumnInfo]
    row_headers: list[str]


@tool
def search_tables(query: str, top_k: int = 5) -> str:
    """Busca tablas relevantes en el índice vectorial."""
    return "[]"


META_PROMPT = """Generá metadatos para esta tabla extraída de un PDF.
Contexto: 'Cuadro 3. Tasa de desocupación por región (en %). Fuente: EPH.'
Columnas: ['Región', '2022', '2023']
Filas:
GBA | 7,1 | 6,5
NOA | 5,2 | 5,0
Patagonia | 4,8 | 4,1"""

CODE_PROMPT = """Escribí un programa Python autocontenido que:
- cree un pandas.DataFrame con columnas ['anio', 'ingresos'] y valores 2021:100, 2022:120, 2023:150,
- calcule el crecimiento interanual promedio en % (promedio simple de las variaciones anuales),
- imprima como ÚLTIMA línea un JSON: {"value": <float>, "unit": "%"}.
Devolvé solo el código en un bloque ```python."""


def run_code(code: str) -> dict:
    """Corre el código en un subproceso aislado (dir temporal, env mínimo, timeout)."""
    with tempfile.TemporaryDirectory() as tmp:
        path = pathlib.Path(tmp) / "main.py"
        path.write_text(code, encoding="utf-8")
        env = {"PATH": os.environ.get("PATH", ""), "SYSTEMROOT": os.environ.get("SYSTEMROOT", "")}
        proc = subprocess.run([sys.executable, "-I", str(path)], cwd=tmp, capture_output=True,
                              text=True, timeout=30, env=env)
    lines = proc.stdout.strip().splitlines()
    return {"returncode": proc.returncode, "last_line": lines[-1] if lines else "", "stderr": proc.stderr[-300:]}


def evaluate(model: str) -> dict:
    kw = {"temperature": 0} if chat_results[model].get("temperature_0") else {}
    llm = make_llm(model, **kw)
    r = {"model": model}
    for method in ("json_schema", "function_calling"):
        t0 = time.perf_counter()
        try:
            out = llm.with_structured_output(TableMetadata, method=method).invoke(META_PROMPT)
            cols_ok = [c.name for c in out.columns] == ["Región", "2022", "2023"]
            r[method] = "OK" if cols_ok else "columnas mal"
            r[f"{method}_s"] = round(time.perf_counter() - t0, 1)
            if method == "json_schema":
                r["ejemplo_metadata"] = out
        except Exception as e:
            r[method] = short_err(e)
    try:
        msg = llm.bind_tools([search_tables]).invoke("Buscá tablas sobre desempleo por región (top 3).")
        r["tools"] = "OK" if msg.tool_calls else "no llamó"
    except Exception as e:
        r["tools"] = short_err(e)
    try:
        t0 = time.perf_counter()
        reply = llm.invoke(CODE_PROMPT).text
        r["codegen_s"] = round(time.perf_counter() - t0, 1)
        m = re.search(r"```(?:python|py)?\s*\n(.*?)```", reply, re.S)
        r["codigo"] = m.group(1) if m else reply
        ex = run_code(r["codigo"])
        value = json.loads(ex["last_line"])["value"]
        r["codegen"] = "OK" if abs(value - 22.5) < 0.01 else f"valor {value}"
    except Exception as e:
        r["codegen"] = short_err(e)
    return r


eval_results = {}
for model in CANDIDATES:
    eval_results[model] = evaluate(model)
    shown = {k: v for k, v in eval_results[model].items() if k not in ("codigo", "ejemplo_metadata")}
    print(shown)

### 6.1 Ejemplos de salida (primer candidato)

In [ ]:
if CANDIDATES:
    first = eval_results[CANDIDATES[0]]
    if first.get("ejemplo_metadata"):
        print(first["ejemplo_metadata"].model_dump_json(indent=2))
    print("\n--- código generado ---\n")
    print(first.get("codigo", ""))

## 7. Resumen → valores para el `.env`

Criterio para el embedding: `text-embedding-3-small` (1536 dims) separa bien relacionadas/no relacionadas, es más barato
y ocupa la mitad en Pinecone que `-large`. `ada-002` es el modelo viejo y separa peor.
Criterio para el chat: el primer candidato que pasa todas las pruebas (orden de `CANDIDATES` ≈ costo creciente).

In [ ]:
working_emb = {
    m: next(r["dim"] for r in res.values() if r.get("ok"))
    for m, res in emb_results.items() if any(r.get("ok") for r in res.values())
}
needs_no_ctx_check = [m for m, res in emb_results.items()
                      if res["check_ctx=False"].get("ok") and not res["check_ctx=True"].get("ok")]
passing = [m for m, r in eval_results.items()
           if all(r.get(k) == "OK" for k in ("json_schema", "function_calling", "tools", "codegen"))]

print("Modelos de chat que responden:", working_chat)
print("Sin soporte de temperature=0:", [m for m in working_chat if not chat_results[m].get("temperature_0")])
print("Candidatos que pasan todas las pruebas:", passing)
print("Modelos de embeddings y dimensión:", working_emb)
print("Embeddings que requieren check_embedding_ctx_length=False:", needs_no_ctx_check)
print()

llm_choice = passing[0] if passing else None
emb_choice = next((m for m in ("text-embedding-3-small", "text-embedding-3-large") if m in working_emb),
                  next(iter(working_emb), None))
print("# --- sugerencia para .env ---")
print(f"LLM_MODEL={llm_choice}")
print(f"EMBEDDING_MODEL={emb_choice}")
print(f"EMBEDDING_DIM={working_emb.get(emb_choice)}")